In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def create_spark_session(app_name: str = "Incrementality") -> SparkSession:

    return (
        SparkSession.builder
        .appName(app_name)
        # .config(...)  # cluster / warehouse configs if needed
        .getOrCreate()
    )

start_date = "2025-01-01"
end_date   = "2025-12-31"
recency_anchor = F.to_date(F.lit("2026-01-01"))
snapshot_date_id = F.date_format(recency_anchor, "yyyyMMdd").cast("int")
start_2024 = "2024-01-01"
end_2024   = "2024-12-31"


T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_rfm = spark.table("cpx_dataanalytics_gold.customer360.c360_f_rfm_model")
T_customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")

In [0]:
pip install openpyxl

In [0]:
import pandas as pd

df = pd.read_excel("/Workspace/Users/ranjan.mishra@cineplex.com/Codebase/CineClub/Offers Post Mortems/Q4 Offer Incementality/DEC2024_breakdown.xlsx")

df_population = spark.createDataFrame(df)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# priority mapping
df_dedup = (
    df_population
    .withColumn(
        "priority",
        F.when(F.col("Category") == "Net_New", 1)
         .when(F.col("Category") == "Renewals", 2)
         .when(F.col("Category") == "Returning", 3)
         .when(F.col("Category") == "Gift-Plan", 4)
         .otherwise(99)
    )
)

w = Window.partitionBy("customers_id").orderBy(F.col("priority"))

df_population_dedup = (
    df_dedup
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "priority")
    .withColumnRenamed("renewal_status", "offer_renewal_status")
)


In [0]:
display(df_population_dedup)

In [0]:
df_pop_temp = (
    df_population_dedup
    .join(T_subscriber.alias("s")
        .withColumnRenamed("renewal_status", "snapshot_renewal_status")
        .withColumnRenamed("plan_id", "snapshot_plan_id"), 
        df_population_dedup.customers_id == T_subscriber.cc_user_profile_id, "inner")
    .filter(F.col("s.snapshot_date_id") == "20260101")
    .withColumnRenamed("Category", "Offer_Category")
    .select(
        "cde_id",
        "Offer_Category",
        "snapshot_renewal_status",
        "snapshot_plan_id",
        "recognitions_redeemed",
        "recognitions_earned",
        "member_priced_recognitions_redeemed",
        "plan_id",
        "offer_renewal_status"
    )
)

In [0]:
df_pop_temp.groupBy("Offer_Category").agg(F.count(F.col("cde_id"))).display()

In [0]:
df_transactions = (
    T_trans.alias("t")
    .join(T_date.select("dateid","datefull"), T_trans.VisitDateId == T_date.dateid, "inner")
     .join(
        T_sessions
        .select("sessionskey", "Auditorium_Experience_ID")
        .alias("s"),
        F.col("t.SessionSkey") == F.col("s.sessionskey"),
        "left"
    )
    .join(
        T_locations
        .select("LocationID", "LC_Location_Type_Description")
        .alias("l"),
        F.col("t.LocationId") == F.col("l.LocationID"),
        "left"
    )
    .filter(F.col("datefull").between("2024-01-01", "2025-12-31"))
    .filter(F.col("t.CDE_ID").isNotNull())
    .withColumn("Visit_Year", F.year(F.col("datefull")))
    .filter(F.col("Visit_Year").isin(2024, 2025))
    .withColumn("CDE_Visit_ID_MNL", F.concat_ws("|",
        F.col("t.CDE_ID").cast("string"),
        F.col("t.VisitDateId").cast("string"),
        F.col("t.LocationId").cast("string")
        )
    )
    .filter(F.col("l.LC_Location_Type_Description") == F.lit("Cineplex Theatre"))
    .groupBy(F.col("t.CDE_ID"), "Visit_Year")
    .agg(
        F.countDistinct(
            F.when(F.col("t.Net_Revenue") > 0, F.col("CDE_Visit_ID_MNL"))
        ).alias("No_of_Visits"),

        F.countDistinct(
            F.when(
                (F.col("t.Net_Revenue") > 0) & (F.col("t.ConcessionFLAG") == "Yes"),
                F.col("CDE_Visit_ID_MNL")
            )
        ).alias("Visits_with_Concession"),
        
        F.sum(F.col("t.Net_Revenue")).alias("Net_Revenue"),

        F.sum(
            F.when(
                (F.col("s.Auditorium_Experience_ID").isin([1, 5, 6])) &
                (F.col("t.TicketFLAG") == "Yes"),
                F.col("t.Quantity")
            ).otherwise(F.lit(0))
        ).alias("No_of_Regular_Tickets"),

        F.sum(
            F.when(
                (F.col("s.Auditorium_Experience_ID").isin([2, 3, 4, 7, 8])) &
                (F.col("t.TicketFLAG") == "Yes"),
                F.col("t.Quantity")
            ).otherwise(F.lit(0))
        ).alias("No_of_Premium_Tickets"),

        F.sum(
            F.when(F.col("t.ConcessionFLAG") == "Yes", F.col("t.Net_Revenue"))
             .otherwise(F.lit(0))
        ).alias("Net_Revenue_Concession"),

        F.sum(
            F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Net_Revenue"))
             .otherwise(F.lit(0))
        ).alias("Net_Revenue_Tickets")
    
    )
)

df_tickets_2025 = (
    T_trans
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("CineClubFLAG") == "Yes")
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .filter(F.col("datefull").between("2025-01-01", "2025-12-31"))
    .groupBy("CDE_ID")
    .agg(
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 6, F.col("Quantity"))).alias("Member_Tickets"),
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 5, F.col("Quantity"))).alias("Companion_Tickets"),
    )
)

df_tickets_2024 = (
    T_trans
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("CineClubFLAG") == "Yes")
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .filter(F.col("datefull").between("2024-01-01", "2024-12-31"))
    .groupBy("CDE_ID")
    .agg(
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 6, F.col("Quantity"))).alias("Member_Tickets"),
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 5, F.col("Quantity"))).alias("Companion_Tickets"),
    )
)

In [0]:
df_out1 = (
    df_pop_temp.alias("p")
    .join(
        df_transactions.alias("t"),
        F.col("p.cde_id") == F.col("t.CDE_ID"),
        "left"
    )
    .join(
        df_tickets_2025.alias("tk25"),
        (F.col("p.cde_id") == F.col("tk25.CDE_ID")) &
        (F.col("t.Visit_Year") == 2025),
        "left"
    )
    .join(
        df_tickets_2025.alias("tk24"),
        (F.col("p.cde_id") == F.col("tk24.CDE_ID")) &
        (F.col("t.Visit_Year") == 2024),
        "left"
    )
    .select(
        F.col("p.Offer_Category").alias("Offer_Category"),
        F.col("t.CDE_ID").alias("CDE_ID"),
        F.col("p.offer_renewal_status").alias("offer_renewal_status"),
        F.col("p.snapshot_renewal_status").alias("snapshot_renewal_status"),
        F.col("p.plan_id").alias("plan_id"),
        F.col("p.snapshot_plan_id").alias("snapshot_plan_id"),
        F.col("t.Visit_Year").alias("Visit_Year"),
        F.col("t.No_of_Visits").alias("No_of_Visits"),
        F.col("t.Visits_with_Concession").alias("Visits_with_Concession"),
        F.col("t.Net_Revenue").alias("Net_Revenue"),
        F.col("t.No_of_Regular_Tickets").alias("No_of_Regular_Tickets"),
        F.col("t.No_of_Premium_Tickets").alias("No_of_Premium_Tickets"),
        F.col("t.Net_Revenue_Concession").alias("Net_Revenue_Concession"),
        F.col("t.Net_Revenue_Tickets").alias("Net_Revenue_Tickets"),

        F.coalesce(F.col("tk25.Member_Tickets"), F.col("tk24.Member_Tickets")).alias("Member_Tickets"),
        F.coalesce(F.col("tk25.Companion_Tickets"), F.col("tk24.Companion_Tickets")).alias("Companion_Tickets")
    )
)


In [0]:
df_out1.groupBy("Offer_Category", "snapshot_plan_id", "snapshot_renewal_status").agg(F.countDistinct("CDE_ID")).display()

In [0]:
df_out1_agg = (
    df_out1
    .groupBy("Offer_Category", "Visit_Year")
    .agg(
        F.countDistinct("CDE_ID").alias("No_of_members"),

        F.sum("No_of_Visits").alias("No_of_Visits"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession"),
        F.sum("Net_Revenue").alias("Net_Revenue"),

        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets"),

        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets"),

        F.sum("Member_Tickets").alias("Member_Tickets"),
        F.sum("Companion_Tickets").alias("Companion_Tickets")
    )
)

display(df_out1_agg)


In [0]:
def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)


In [0]:
df_out1_kpis = (
    df_out1_agg
    .withColumn("Avg Frequency", sdiv(F.col("No_of_Visits"), F.col("No_of_members")))
    .withColumn("Avg Total Spend per visit", sdiv(F.col("Net_Revenue"), F.col("No_of_Visits")))
    .withColumn("Avg Ticket Spend per visit", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Avg Concession Spend per visit", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Spend per person in a year", sdiv(F.col("Net_Revenue"), F.col("No_of_members")))
    .withColumn("Ticket Spend per person in a year", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Spend per person in a year", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Incidence", sdiv(F.col("Visits_with_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Tickets per Visit", sdiv(F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets"), F.col("No_of_Visits")))
    .withColumn("999 Tickets per Visit", sdiv(F.col("Member_Tickets") + F.col("Companion_Tickets"), F.col("No_of_Visits")))
    .withColumn(
        "Uptake Premium Experiences",
        sdiv(
            F.col("No_of_Premium_Tickets"),
            (F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets"))
        )
    )
    .withColumn(
        "Uptake Companion Experiences",
        sdiv(F.col("Companion_Tickets"), F.col("Member_Tickets"))
    )
    .filter(F.col("Visit_Year").isNotNull())
)

display(df_out1_kpis)